# Evidently client demo

This notebook runs in a plain Docker container (see `../README.md`), computes an Evidently drift report **locally**, then pushes the resulting snapshot to the `evidently-server` pod running in the cluster, over the network via `RemoteWorkspace`.

All API calls below are the current (non-legacy) Evidently API, verified directly against the installed `evidently==0.7.21` source: `evidently/core/report.py` (`Report.run()` returns a `Snapshot`) and `evidently/ui/workspace.py` (`RemoteWorkspace`, `create_project`, `add_run`).

In [ ]:
import os

import numpy as np
import pandas as pd

from evidently import Dataset, DataDefinition, Report
from evidently.presets import DataDriftPreset
from evidently.ui.workspace import RemoteWorkspace

EVIDENTLY_SERVER_URL = os.environ["EVIDENTLY_SERVER_URL"]
EVIDENTLY_SERVER_URL

## 1. Generate reference and current data

`reference` is the "known good" baseline distribution; `current` is a batch deliberately shifted on `feature_1` so the drift report has something real to detect.

In [ ]:
rng = np.random.default_rng(42)

reference = pd.DataFrame(
    {
        "feature_1": rng.normal(loc=0, scale=1, size=500),
        "feature_2": rng.normal(loc=5, scale=2, size=500),
    }
)

current = pd.DataFrame(
    {
        "feature_1": rng.normal(loc=3, scale=1, size=500),  # shifted mean -> should trigger drift
        "feature_2": rng.normal(loc=5, scale=2, size=500),
    }
)

reference.describe()

## 2. Run the report locally

`Dataset.from_pandas(..., data_definition=DataDefinition())` auto-infers column types; `Report.run()` computes the report and returns a `Snapshot` -- the same object type `RemoteWorkspace.add_run()` expects, so nothing needs converting.

In [ ]:
reference_dataset = Dataset.from_pandas(reference, data_definition=DataDefinition())
current_dataset = Dataset.from_pandas(current, data_definition=DataDefinition())

report = Report([DataDriftPreset()])
snapshot = report.run(current_dataset, reference_dataset)
snapshot

## 3. Send it to the remote server

`RemoteWorkspace` talks to `evidently-server`'s HTTP API. `create_project` is idempotent-ish for this demo -- re-running the cell just makes a second project; in real use you'd look it up first with `list_projects()`/`search_project()`.

In [ ]:
workspace = RemoteWorkspace(EVIDENTLY_SERVER_URL)

project = workspace.create_project(
    "k8n-mlops-demo", description="Sent from the jupyter_client Docker container"
)

workspace.add_run(project.id, snapshot, include_data=False)
print(f"Uploaded snapshot to project {project.id} ({project.name})")

## 4. View it

Open `EVIDENTLY_SERVER_URL` in a browser -- the project and this run should now be visible in the Evidently UI, computed here in the Jupyter container but stored and rendered entirely by the server pod.